# 03 — Historical Churn Feature Engineering

## Project
Netflix Customer Intelligence & AI Analytics Copilot

## Objective
Construct point-in-time customer snapshots for predicting churn during the
following 30 days.

The notebook executes the existing scripts:
- `src/features/build_churn_features.py`
- `src/features/build_advanced_churn_features.py`

The historical cutoffs are January–May 2026. Base features use viewing activity
from the preceding 90 days and payment/support history through each cutoff.
Advanced features compare the latest 30 days with the preceding 30 days.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "features" / "build_churn_features.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_DIR = PROJECT_ROOT / "data" / "features"
import os
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Feature directory:", FEATURE_DIR)


Project root: /workspaces/Netflix-Customer-Intelligence
Feature directory: /workspaces/Netflix-Customer-Intelligence/data/features


## 2. Build historical snapshots

In [2]:
%run src/features/build_churn_features.py

2026-01-31: rows=6,770, positive=202, negative=6,568


2026-02-28: rows=6,681, positive=206, negative=6,475


2026-03-31: rows=6,571, positive=218, negative=6,353


2026-04-30: rows=6,427, positive=232, negative=6,195


2026-05-31: rows=6,260, positive=251, negative=6,009



Saved: /workspaces/Netflix-Customer-Intelligence/data/features/churn_historical_snapshots.csv
Total snapshot rows: 32,709
Unique customers across all snapshots: 7,116

Label status: provisional. Verify complete 30-day follow-up before model training.


## 3. Build advanced behavioural features

In [3]:
%run src/features/build_advanced_churn_features.py


Processing snapshot: 2026-01-31


Customers: 6,770 | Declining engagement: 2,032 | Payment risk: 85 | Support risk: 42

Processing snapshot: 2026-02-28


Customers: 6,681 | Declining engagement: 2,030 | Payment risk: 57 | Support risk: 32

Processing snapshot: 2026-03-31


Customers: 6,571 | Declining engagement: 2,268 | Payment risk: 65 | Support risk: 46

Processing snapshot: 2026-04-30


Customers: 6,427 | Declining engagement: 1,843 | Payment risk: 71 | Support risk: 43

Processing snapshot: 2026-05-31


Customers: 6,260 | Declining engagement: 1,495 | Payment risk: 61 | Support risk: 47



ADVANCED FEATURE ENGINEERING COMPLETED
Output: /workspaces/Netflix-Customer-Intelligence/data/features/churn_advanced_snapshots.csv
Dataset shape: (32709, 61)
Unique customers: 7116
Duplicate customer-snapshot pairs: 0
Target distribution:
 churn_next_30_days
0    31600
1     1109


## 4. Load and inspect engineered dataset

In [4]:
base_path = FEATURE_DIR / "churn_historical_snapshots.csv"
advanced_path = FEATURE_DIR / "churn_advanced_snapshots.csv"

base = pd.read_csv(base_path)
features = pd.read_csv(advanced_path)

print("Base shape:", base.shape)
print("Advanced shape:", features.shape)
print("Unique customers:", features["customer_id"].nunique())
print("Duplicate customer-snapshot pairs:",
      features.duplicated(["customer_id", "snapshot_date"]).sum())

display(features.head())


Base shape: (32709, 28)
Advanced shape: (32709, 61)
Unique customers: 7116
Duplicate customer-snapshot pairs: 0


,customer_id,age,gender,country,city,preferred_language,plan_id,churn_next_30_days,viewing_sessions_90d,active_viewing_days_90d,...,has_unresolved_support_30d,has_critical_support_30d,watch_minutes_change_pct,viewing_sessions_change_pct,active_days_change_pct,payment_failures_change,support_ticket_change,declining_engagement_flag,payment_risk_flag,support_risk_flag
0,CUST000744,21.0,Female,United Kingdom,Liverpool,English,PLAN03,0,1.0,1.0,...,0,0,100.0,100.0,100.0,0.0,0.0,0,0,0
1,CUST001502,31.0,Male,Canada,Montreal,French,PLAN01,0,1.0,1.0,...,0,0,100.0,100.0,100.0,0.0,0.0,0,0,0
2,CUST005383,40.0,Male,U.K.,Leeds,English,PLAN01,0,5.0,5.0,...,0,0,100.0,100.0,100.0,0.0,0.0,0,0,0
3,CUST005877,36.0,Male,Indonesia,Jakarta,French,PLAN03,0,2.0,2.0,...,0,0,3200.0,0.0,0.0,-1.0,0.0,0,0,0
4,CUST003762,29.0,Male,Poland,Warsaw,English,PLAN02,0,7.0,7.0,...,0,0,100.0,100.0,100.0,1.0,0.0,0,1,0


## 5. Target distribution and snapshot coverage

In [5]:
target_summary = (
    features.groupby("snapshot_date")["churn_next_30_days"]
    .agg(rows="size", churned="sum", churn_rate="mean")
    .reset_index()
)
display(target_summary)

print("\nOverall target distribution:")
display(features["churn_next_30_days"].value_counts(dropna=False))


,snapshot_date,rows,churned,churn_rate
0,2026-01-31,6770,202,0.029838
1,2026-02-28,6681,206,0.030834
2,2026-03-31,6571,218,0.033176
3,2026-04-30,6427,232,0.036098
4,2026-05-31,6260,251,0.040096



Overall target distribution:


churn_next_30_days
0    31600
1     1109
Name: count, dtype: int64

## 6. Feature groups

In [6]:
feature_groups = {
    "Customer profile": [
        "age", "gender", "country", "city", "preferred_language"
    ],
    "Viewing behaviour": [
        "viewing_sessions_90d", "active_viewing_days_90d",
        "total_watch_minutes_90d", "avg_watch_minutes_90d",
        "avg_completion_pct_90d", "days_since_last_viewing",
        "viewing_sessions_30d", "watch_minutes_30d",
        "watch_minutes_previous_30d", "watch_minutes_change_pct",
        "viewing_sessions_change_pct", "active_days_change_pct"
    ],
    "Payment behaviour": [
        "payment_count", "failed_payment_count",
        "successful_payment_count", "failed_payment_rate",
        "payments_30d", "failed_payments_30d",
        "failed_payment_rate_30d", "payment_failures_change"
    ],
    "Support behaviour": [
        "support_ticket_count", "high_priority_ticket_count",
        "unresolved_ticket_count", "avg_support_satisfaction",
        "avg_resolution_hours", "support_tickets_30d",
        "unresolved_tickets_30d", "escalated_tickets_30d"
    ],
    "Risk indicators": [
        "declining_engagement_flag", "payment_risk_flag",
        "support_risk_flag", "has_unresolved_support_30d",
        "has_critical_support_30d"
    ]
}

for group, columns in feature_groups.items():
    present = [column for column in columns if column in features.columns]
    print(f"{group}: {len(present)} available columns")
    print(", ".join(present))


Customer profile: 5 available columns
age, gender, country, city, preferred_language
Viewing behaviour: 12 available columns
viewing_sessions_90d, active_viewing_days_90d, total_watch_minutes_90d, avg_watch_minutes_90d, avg_completion_pct_90d, days_since_last_viewing, viewing_sessions_30d, watch_minutes_30d, watch_minutes_previous_30d, watch_minutes_change_pct, viewing_sessions_change_pct, active_days_change_pct
Payment behaviour: 8 available columns
payment_count, failed_payment_count, successful_payment_count, failed_payment_rate, payments_30d, failed_payments_30d, failed_payment_rate_30d, payment_failures_change
Support behaviour: 8 available columns
support_ticket_count, high_priority_ticket_count, unresolved_ticket_count, avg_support_satisfaction, avg_resolution_hours, support_tickets_30d, unresolved_tickets_30d, escalated_tickets_30d
Risk indicators: 5 available columns
declining_engagement_flag, payment_risk_flag, support_risk_flag, has_unresolved_support_30d, has_critical_suppo

## 7. Data-quality and leakage checks

The source pipeline uses historical date filters for viewing and payment
events. However, support-ticket status fields represent current recorded
statuses. If those statuses were updated after a snapshot cutoff, the resulting
support features may contain information unavailable at prediction time.

This is a leakage risk that requires investigation before claiming that the
model is fully point-in-time safe.

The 30-day churn target also requires complete follow-up observation. The
source script labels snapshots as provisional and explicitly asks for this
verification.


In [7]:
required = ["customer_id", "snapshot_date", "churn_next_30_days"]
missing_required = [c for c in required if c not in features.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

checks = {
    "duplicate_customer_snapshot_pairs":
        int(features.duplicated(["customer_id", "snapshot_date"]).sum()),
    "missing_customer_ids": int(features["customer_id"].isna().sum()),
    "missing_snapshot_dates": int(features["snapshot_date"].isna().sum()),
    "invalid_target_values":
        int((~features["churn_next_30_days"].isin([0, 1])).sum())
}
display(pd.Series(checks, name="count").to_frame())


,count
duplicate_customer_snapshot_pairs,0
missing_customer_ids,0
missing_snapshot_dates,0
invalid_target_values,0


## 8. Feature-engineering conclusions

The project creates historical customer snapshots and adds behavioural
features for engagement decline, payment friction and support risk.

The engineered dataset is suitable for further model experimentation, subject
to:
1. Verifying full 30-day outcome follow-up for every snapshot.
2. Investigating support-status temporal leakage.
3. Keeping customer identifiers and snapshot metadata out of model predictors.
4. Performing model validation that accounts for repeated customers across
   historical snapshots.

Feature importance and model performance indicate predictive association,
not causal explanations of churn.
